# Tramontane — GDPR-Compliant Agent Pipelines

This notebook demonstrates Tramontane's GDPR-native features:
PII detection, data redaction, Article 30 report generation,
and EU-sovereign deployment.

Built in Orléans, France 🇫🇷 · [github.com/bleucommerce/tramontane](https://github.com/bleucommerce/tramontane)

In [ ]:
!pip install tramontane -q

## 1. PII Detection — French Patterns

Tramontane detects French PII out of the box: emails, phone numbers (+33),
IBAN (FR), NIR (sécurité sociale), passport numbers, and named entities.

In [ ]:
from tramontane.gdpr.pii import PIIDetector

detector = PIIDetector()  # OFFLINE mode, no API key needed

samples = [
    "Envoyez le contrat \u00e0 jean.dupont@enterprise.fr",
    "Appelez Mme Martin au 06 12 34 56 78",
    "IBAN: FR76 3000 6000 0112 3456 7890 189",
    "Le temps \u00e0 Paris est agr\u00e9able aujourd'hui",
]

for text in samples:
    result = detector.detect_sync(text)
    pii_list = [t.value for t in result.pii_types_found]
    status = f"\u26a0\ufe0f {pii_list}" if result.has_pii else "\u2705 Clean"
    print(f"{text[:50]:<52} {status}")
    if result.has_pii:
        print(f"  Redacted: {result.cleaned_text}")

## 2. GDPR Middleware Levels

Three enforcement levels:
- `none`: passthrough, audit only
- `standard`: detect PII, log, pass through
- `strict`: detect PII, **redact**, log, block if needed

In [ ]:
import asyncio

from tramontane.gdpr.middleware import GDPRMiddleware

text = "Send the invoice to jean.dupont@email.fr, +33 6 12 34 56 78"

for level in ["none", "standard", "strict"]:
    mw = GDPRMiddleware(gdpr_level=level)
    result = asyncio.run(mw.process_input(
        text=text, run_id="demo", agent_role="test"
    ))
    print(f"[{level:8s}] {result}")

## 3. Document Analysis Pipeline (GDPR Strict)

A three-agent pipeline that processes documents with `gdpr_level=strict`.
All PII is automatically redacted before each agent sees the content.

In [ ]:
from tramontane.core.pipeline import Pipeline

pipeline = Pipeline.from_yaml("pipelines/document_analysis.yaml")

print(f"Pipeline:   {pipeline.name}")
print(f"GDPR level: {pipeline.gdpr_level}")
print(f"Budget:     \u20ac{pipeline.budget_eur}")
print(f"Agents:     {list(pipeline.agents.keys())}")
print(f"\n{pipeline.handoff_graph.to_mermaid()}")

## 4. Article 30 Report Generation

GDPR Article 30 requires controllers to maintain a record of processing activities.
Tramontane generates these automatically from the audit vault.

In [ ]:
import json

from tramontane.gdpr.reports import GDPRReporter

reporter = GDPRReporter()
report = reporter.article_30_report_sync()

print(json.dumps(report, indent=2, default=str, ensure_ascii=False))

## 5. Right to Erasure (Article 17)

Users can request deletion of their data. Tramontane soft-deletes
memory entries (sets `erased_at`), removes from FTS index, and
logs the erasure event. Audit trail is preserved (append-only).

In [ ]:
import os
import tempfile

from tramontane.memory.longterm import LongTermMemory

db_path = os.path.join(tempfile.gettempdir(), "demo_gdpr.db")
mem = LongTermMemory(db_path=db_path)

# Store some user data
asyncio.run(mem.store(
    content="User prefers French communication",
    entity_key="prefs", memory_type="preference", user_id="user_42"
))

# User requests erasure
count = asyncio.run(mem.erase_user("user_42"))
print(f"Erased {count} entries for user_42")

# Verify: search returns nothing
results = asyncio.run(mem.search("French", user_id="user_42"))
print(f"Search after erasure: {len(results)} results")

os.unlink(db_path)

## EU Deployment

Tramontane is designed for EU-sovereign deployment:

- **Models**: Mistral AI (Paris, France)
- **Infrastructure**: Scaleway EU-west-1 (Paris)
- **Data residency**: Data never leaves EU
- **HTTP 451**: `GDPRViolationError` returns the legally correct status code
- **X-EU-Sovereign: true** header on all API responses

```bash
docker compose up -d
# API at http://localhost:8080
# Docs at http://localhost:8080/docs
```